# Phase 4 — AI/ML Layer: Uniswap Tokenomics

**Reproducible (DEL-03):** this notebook runs top-to-bottom from the cached `data/*.csv` snapshots with **no Dune key** and **no network call**. It imports the validated companion `notebooks/04_ml_analysis.py` so the numbers and figures here reproduce that script exactly (`random_state=42`).

**Additive & cuttable:** this layer only *sharpens* the Phase-3 verdict ("real but modest & unproven"); it reads Phase 1-3 artifacts as inputs only and deleting it breaks no Phase 1-3 / verdict claim (the delete-the-AI-section test).

Three outputs, each ending in a **"Sharpens the verdict:"** sentence:
1. AI-02 — wallet clustering (KMeans)
2. AI-03 — anomaly detection (IsolationForest)
3. AI-04 — buy-and-burn break-even model

## 1. Setup / load
Import the validated companion module and the cached holder/entity data. All paths are repo-relative `data/` so this runs in Colab/Jupyter from the repo.

In [1]:
import importlib.util
from pathlib import Path

# Locate the repo root from the notebook's location (notebooks/ -> parent).
REPO_ROOT = Path.cwd()
if (REPO_ROOT / 'notebooks').exists() and (REPO_ROOT / 'data').exists():
    pass  # launched from repo root
elif REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
else:
    # Fallback: search upward for a dir containing data/ and notebooks/.
    p = REPO_ROOT
    while p != p.parent and not ((p / 'data').exists() and (p / 'notebooks').exists()):
        p = p.parent
    REPO_ROOT = p

_spec = importlib.util.spec_from_file_location(
    'ml04', REPO_ROOT / 'notebooks' / '04_ml_analysis.py')
ml04 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ml04)

ml04._ensure_figures_dir()
df = ml04.load_holder_features()
entities = ml04.load_entity_labels()
as_of = ml04.holder_features_as_of()
print('holder_features rows:', len(df))
print('columns:', list(df.columns))
print('data-as-of:', as_of)

holder_features rows: 1000
columns: ['addr', 'balance_uni', 'pct_of_supply', 'n_transfers', 'total_in_uni', 'total_out_uni', 'first_seen', 'last_seen', 'active_span_days', 'days_since_last']
data-as-of: 2026-06-21T22:47:32Z


## 2. Clustering (AI-02)
KMeans on log-scaled, StandardScaler-normalized behavioral features of the top-1000 UNI holders; silhouette-driven `k`; honest centroid-labeled archetypes; PCA scatter; treasury/CEX spot-check. Figures: `analysis/figures/silhouette.png`, `clusters.png`.

In [2]:
clustering = ml04.run_clustering(df, entities, as_of)


=== AI-02 Wallet Clustering (top-1000 UNI holders) ===
data-as-of: 2026-06-21T22:47:32Z

Silhouette vs k:
    k  silhouette
    3      0.2943
    4      0.2782
    5      0.2947
    6      0.2977  <- chosen

Chosen k = 6  (silhouette = 0.2977)

CAVEAT (silhouette ≈ 0.30): clústeres útiles pero no nítidamente separados — leer las etiquetas (whale / active / retail-high / retail-mid / retail-low / dormant) como segmentos interpretativos, no como categorías económicas duras.



Nota de mapeo (honesta): whale = mayor saldo; active = mayor nº de transferencias; dormant = mayor inactividad; los tres tramos restantes se ordenan en retail-high/mid/low por saldo descendente. Cuando dos tramos tienen saldo casi idéntico, el desempate asigna retail-mid al más activo (mayor nº de transferencias) y retail-low al menos activo.

Centroid / archetype table (interpretable units):
     archetype  size  balance_uni  n_transfers  active_span_days  days_since_last  out_in_ratio
0       active   106   256,837.19     3,570.05          1,539.75             4.45          0.87
1        whale   161 1,264,890.38         6.20          1,015.04           210.56          0.15
2  retail-high   166    91,032.84       139.14            455.52            24.43          0.72
3   retail-low   189    54,272.40         7.44            187.14            47.82          0.11
4   retail-mid   117    54,292.24        17.33          1,582.74           188.16          0.47
5      dormant   261    51,


Figures written:
  /Users/jaimeberdejosanchez/projects/ProyectoBlockchain/analysis/figures/silhouette.png
  /Users/jaimeberdejosanchez/projects/ProyectoBlockchain/analysis/figures/clusters.png

Sharpens the verdict: the holder base separates into distinct behavioral archetypes (active, dormant, retail-high, retail-low, retail-mid, whale), with the high-balance custodial/treasury 'whale' cluster (16.1% of the top-1000 by count but the dominant share of supply) sitting apart from more dispersed active/retail holders — so burn-based value capture accrues pro-rata to supply, meaning it concentrates in a few large custodial/treasury wallets rather than meaningfully reaching dispersed holders, which keeps the verdict 'real but modest & unproven.'
Scope note: this clusters only the top-1000 holders (the economically meaningful supply); true long-tail retail/dormant below rank 1000 and on-DEX LP-position detection are NOT in this snapshot — a stated limitation, not a claim.


## 3. Anomaly detection (AI-03)
IsolationForest (`random_state=42`) on (a) the **same** holder feature matrix -> flagged wallets each mapped to a stated insight and cross-referenced to entity labels; and (b) the weekly burn series with the **2025-12-22 100M-burn week whitelisted**, reporting any OTHER anomalous weeks. Figure: `analysis/figures/anomalies.png`.

In [3]:
anomaly = ml04.run_anomaly_detection(df, entities, as_of)


=== AI-03 Anomaly Detection (IsolationForest) ===
data-as-of: 2026-06-21T22:47:32Z



Wallet anomalies: 20 of 1000 top-1000 holders flagged (contamination=2%, IsolationForest random_state=42).

Flagged wallets (each ends in a stated insight):
  0x090d461347... [unlabeled] bal=12,543,368 UNI, tx=222,045, out/in=0.92
     insight: extreme transfer count (222,045) at modest balance — likely an unlabeled exchange/router hot wallet, not an organic holder
  0x1a9c8182c0... [Uniswap Timelock/Treasury] bal=272,134,858 UNI, tx=135, out/in=0.39
     insight: labeled 'Uniswap Timelock/Treasury' flagged on extreme activity (135 transfers) — EXPECTED custodial churn, not noise
  0x1d42064fc4... [unlabeled] bal=3,206,546 UNI, tx=520,178, out/in=1.00
     insight: extreme transfer count (520,178) at modest balance — likely an unlabeled exchange/router hot wallet, not an organic holder
  0x28c6c06298... [Binance (CEX)] bal=6,043,600 UNI, tx=262,889, out/in=0.99
     insight: labeled 'Binance (CEX)' flagged on extreme activity (262,889 transfers) — EXPECTED custodial churn, not noise
 

Most unusual judged weeks by the model (n=25, small sample; contamination=10% forces the top ~10%): 3 flagged, excluding the whitelisted 100M week.
  CAVEAT: with this few weekly points this is a ranking/flagging of the largest routine burns, NOT robust statistical anomaly detection — read it as 'the model's top-decile burn weeks by construction', not 'the detector found structure'.
  2025-12-29: 72,003 UNI burned ($424,774) — among the larger routine-burn weeks; magnitude is ~0.1-0.6M UNI, i.e. ROUTINE buy-and-burn, NOT a second structural 100M-scale event
  2026-04-13: 378,000 UNI burned ($1,235,889) — among the larger routine-burn weeks; magnitude is ~0.1-0.6M UNI, i.e. ROUTINE buy-and-burn, NOT a second structural 100M-scale event
  2026-06-08: 600,000 UNI burned ($1,504,932) — among the larger routine-burn weeks; magnitude is ~0.1-0.6M UNI, i.e. ROUTINE buy-and-burn, NOT a second structural 100M-scale event



Figure written:
  /Users/jaimeberdejosanchez/projects/ProyectoBlockchain/analysis/figures/anomalies.png

Sharpens the verdict: the anomaly layer surfaces NO hidden second structural burn beyond the one-time 100M UNIfication event — routine weekly burns are small and orderly — while the flagged wallets are dominated by labeled custodial/CEX churn rather than organic dispersed holders, so it reinforces that value capture is real but modest and concentrated, not a broad reflexive flywheel.


## 4. Break-even (AI-04)
The daily Uniswap volume at which protocol-fee buy-and-burn turns UNI net-deflationary, in two issuance scenarios: (a) current issuance = 0 (headroom) and (b) 2% tail + 20M/yr growth re-enabled (break-even daily volume). Constants are documented and traceable to FOUNDATION.md / headline_metrics.csv. The only price used is the static cached spot for USD->UNI conversion — **no price prediction**. Figure: `analysis/figures/breakeven.png`.

In [4]:
breakeven = ml04.run_breakeven(as_of)


=== AI-04 Buy-and-Burn Break-Even Model ===
data-as-of: 2026-06-21T22:47:32Z

Documented constants (traceable to FOUNDATION.md / headline_metrics.csv):
  LP fee rate                = 0.1900% of volume (uniswap_volume_weekly.csv)
  protocol take (blended)    = 7.83% of LP fees (CALIBRATED so the model reproduces the observed annualized burn at recent volume;
                               per-pool taxonomy: v2 16.7%, v3 1/4 or 1/6 — blended lower because the switch covers only a subset of pools)
    CAVEAT (WR-03): this calibration pins an instantaneous annualized burn to a 12-week-avg
    volume base, so the blended take is an EFFECTIVE calibrated rate per $ volume that absorbs
    any volume-period mismatch — it is NOT a pure protocol-fee carve-out of LP fees.
  UNI price (static spot)    = $3.03 (headline_metrics.csv; USD->UNI only, NOT forecast)
  total supply               = 893,790,420 UNI (headline_metrics.csv)
  tail inflation             = 2%/yr (FOUNDATION M5)
  growth budget


Figure written:
  /Users/jaimeberdejosanchez/projects/ProyectoBlockchain/analysis/figures/breakeven.png

Sharpens the verdict: with issuance currently at 0 the burn is net-deflationary at any positive volume, but if the 2% tail + 20M growth budget were re-enabled the network would need ~$2.1B/day of volume to stay deflationary — roughly 3.0x today's $0.70B/day — so the 'sustainable?' answer is conditional on both volume holding up AND the revocable zero-issuance policy persisting, exactly the net-deflationary-but-conditional framing of the Phase-3 verdict.


## 5. Report context figures (before/after UNIfication)
Spanish-labelled context charts embedded in the report body, all read from the cached `data/*.csv` (no network): weekly volume + burn value (`volume_burn_timeline.png`), UNI vs ETH vs AAVE rebased to 100 at UNIfication (`price_comparison.png`), and Uniswap TVL (`tvl_timeline.png`).

In [5]:
fig_volume_burn = ml04.make_volume_burn_timeline()
fig_price = ml04.make_price_comparison()
fig_tvl = ml04.make_tvl_timeline()


Figure written:
  /Users/jaimeberdejosanchez/projects/ProyectoBlockchain/analysis/figures/volume_burn_timeline.png



Figure written:
  /Users/jaimeberdejosanchez/projects/ProyectoBlockchain/analysis/figures/price_comparison.png



Figure written:
  /Users/jaimeberdejosanchez/projects/ProyectoBlockchain/analysis/figures/tvl_timeline.png


## 6. How this sharpens the verdict
Each output ends in one verdict-sharpening sentence (printed above); restated here:

- **AI-02 (clustering):** the holder base splits into distinct behavioral archetypes, with a high-balance custodial/treasury *whale* cluster apart from dispersed active/retail holders — so pro-rata burn value capture concentrates in a few large wallets rather than meaningfully reaching dispersed holders.
- **AI-03 (anomaly):** no hidden second structural burn beyond the one-time 100M UNIfication event, and the flagged wallets are dominated by labeled custodial/CEX churn — reinforcing that value capture is real but modest and concentrated, not a broad reflexive flywheel.
- **AI-04 (break-even):** net-deflationary at any volume while issuance is 0, but a re-enabled 2% tail + 20M growth budget would require ~3x today's volume to stay deflationary — so sustainability is conditional on volume holding up **and** the revocable zero-issuance policy persisting.

**Honest scope / delete-the-AI-section note:** clustering covers only the top-1000 holders (long-tail and on-DEX LP-position detection are out of this snapshot); the break-even uses a static spot price purely for USD->UNI conversion (no forecast). This whole layer reads Phase 1-3 artifacts as inputs only and asserts **no new verdict** — removing it breaks nothing upstream. It sharpens, never gates, the Phase-3 verdict: *real but modest & unproven.*